# Build the historical repository (Colab)

Runs `scripts/build_repository_index.py` on every `pending` video in the shared database:
FFmpeg normalisation, keyframes, pHash + OpenCLIP into FAISS, Whisper transcripts, OCR.
Raw videos are fetched from the shared Drive folder (`raw_videos/`) and processed files are
archived back to Drive.

**Rules (read before running anything)**
1. **Only one person runs this notebook at a time.** The FAISS index on Drive is last-write-wins.
   Announce "running the build now" in the group chat first; say "done" when it finishes.
2. **Runtime -> Change runtime type -> GPU (T4 is fine)** before the first cell.
3. Run the cells top to bottom. Cell 7 refuses to start until you set `ANNOUNCED_IN_GROUP = True`.

**One-time: add these in Colab Secrets (key icon in the left sidebar), notebook access ON**

| Secret | Value |
|---|---|
| `DATABASE_URL` | the shared Neon connection string (same one in everyone's `.env`) |
| `RCLONE_CLIENT_ID` | the OAuth client ID of the team's Google Cloud app |
| `RCLONE_CLIENT_SECRET` | the OAuth client secret |
| `RCLONE_ROOT_FOLDER_ID` | ID of the shared Drive folder (the part after `/folders/`) |
| `RCLONE_TOKEN` | the **single-line** JSON printed by `rclone authorize "drive" "<client_id>" "<client_secret>"`, run on a laptop with a browser |

In the Secrets panel the **name** goes on the left (e.g. `DATABASE_URL`) and the **value** on the right.
The cells only ever refer to the *name*, like `userdata.get('DATABASE_URL')`. Never paste a value into
a cell or commit it.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Get the code and install dependencies

In [ ]:
import os
if not os.path.isdir('/content/Resurface'):
    !git clone -q https://github.com/nocriteriaforaevaaaaa/Resurface.git /content/Resurface
%cd /content/Resurface
!git checkout -q master && git pull -q
!git log --oneline -3

# openai-whisper's setup.py needs setuptools<81 first (see requirements.txt comment)
!pip install -q "setuptools<81"
!pip install -q --no-build-isolation openai-whisper
!pip install -q -r requirements.txt

# make `import src...` work when running scripts/*.py
%env PYTHONPATH=/content/Resurface

## 3. Write `.env` from Colab Secrets

In [ ]:
from google.colab import userdata

# Only DATABASE_URL is secret; the rest mirror the team's local .env so Colab behaves the same.
# (YOUTUBE_DATA_API_KEY is only needed for discovery, not for processing, so it is omitted.)
env_contents = f"""DATABASE_URL={userdata.get('DATABASE_URL')}
WHISPER_MODEL_NAME=small
OPEN_CLIP_MODEL_NAME=ViT-B-32
OPEN_CLIP_PRETRAINED=laion2b_s34b_b79k
OCR_ENGINE=easyocr
FAISS_TOP_K=50
PHASH_HAMMING_THRESHOLD=10
MIN_FINAL_SCORE_FOR_MATCH=0.55
MIN_SOURCE_RELIABILITY_FOR_DECISION=0.4
TORCH_DEVICE=auto
USE_REMOTE_STORAGE=true
RCLONE_REMOTE=gdrive
RCLONE_REMOTE_ROOT=
"""
with open('.env', 'w') as f:
    f.write(env_contents)
print('.env written (values not shown)')

## 4. Install and configure rclone from Colab Secrets

`RCLONE_REMOTE_ROOT` is empty because the remote's root *is* the shared folder.

In [ ]:
!curl -s https://rclone.org/install.sh | sudo bash

import os
conf = f"""[gdrive]
type = drive
client_id = {userdata.get('RCLONE_CLIENT_ID')}
client_secret = {userdata.get('RCLONE_CLIENT_SECRET')}
scope = drive
root_folder_id = {userdata.get('RCLONE_ROOT_FOLDER_ID')}
token = {userdata.get('RCLONE_TOKEN')}
team_drive =
"""
os.makedirs('/root/.config/rclone', exist_ok=True)
with open('/root/.config/rclone/rclone.conf', 'w') as f:
    f.write(conf)
os.chmod('/root/.config/rclone/rclone.conf', 0o600)

# must list the shared folder's contents (raw_videos, faiss_index, ...)
!rclone lsd gdrive:

## 5. Pre-flight: is every pending video on Drive?

Stops with an error if any pending video's file is missing from `gdrive:raw_videos`,
because the build would just mark those `failed`.

In [ ]:
import re, subprocess
from collections import Counter
from src.db.session import SessionLocal
from src.db.models import Video, ProcessingStatus

res = subprocess.run(['rclone', 'lsf', 'gdrive:raw_videos', '--files-only'],
                     capture_output=True, text=True)
if res.returncode != 0:
    # show rclone's real error (never prints the config/token)
    raise RuntimeError('rclone could not list gdrive:raw_videos:\n' + res.stderr[-1500:])
on_drive = set(res.stdout.split())

db = SessionLocal()
videos = db.query(Video).all()
print('status counts:', dict(Counter(v.status.value for v in videos)))
pending = [v for v in videos if v.status == ProcessingStatus.pending]
# paths were recorded on the downloader's machine (may be Windows): compare by file name
file_name = lambda p: re.split(r'[\\/]', p)[-1]
missing = [v.raw_file_path for v in pending if file_name(v.raw_file_path) not in on_drive]
db.close()

print(f'pending: {len(pending)} | on Drive: {len(pending) - len(missing)} | missing: {len(missing)}')
assert not missing, f'{len(missing)} pending video(s) are not on Drive yet, e.g. {missing[:3]}'
print('OK: every pending video is on Drive')

## 6. Back up the shared FAISS index on Drive

The build pulls the shared index, adds vectors and pushes it back. This keeps a dated copy
in case the run goes wrong. Skip only if `faiss_index` does not exist on Drive yet.

In [ ]:
import datetime
backup = f"faiss_index_backup_{datetime.date.today().isoformat()}"
!rclone copy gdrive:faiss_index gdrive:{backup} --stats-one-line
!rclone lsd gdrive:

## 7. Run the build

Set `ANNOUNCED_IN_GROUP = True` **only after** you told the group you are starting and nobody
else is running a build or a query. Watch the first two or three videos finish (Drive download,
processing, upload) before walking away. If you must stop it, tell the group before anyone
re-runs.

In [ ]:
ANNOUNCED_IN_GROUP = True  # <- set to True once you have announced it

assert ANNOUNCED_IN_GROUP, 'Announce in the group chat first, then set ANNOUNCED_IN_GROUP = True'
!cd /content/Resurface && PYTHONPATH=/content/Resurface python -u scripts/build_repository_index.py

## 8. Summary after the run

Tell the group "done" once this shows no `normalising` / `extracting_features` rows left
over. Videos marked `failed` need investigating before a re-run.

In [ ]:
from collections import Counter
from src.db.session import SessionLocal
from src.db.models import Video

db = SessionLocal()
videos = db.query(Video).all()
print('status counts:', dict(Counter(v.status.value for v in videos)))
for v in videos:
    if v.status.value not in ('indexed', 'pending'):
        print(v.status.value, v.source_url)
db.close()